In [0]:
"""
sim_hr_delta_v2.py
==================
Enhanced simulation script: generates synthetic HR data and writes it
into Databricks Delta tables under smart_odoo.silver

Tables covered (3 total):
    1. silver.hr_department   — fixed 5 departments; seeded once, skipped on re-runs
    2. silver.hr_job          — fixed 10 jobs;        seeded once, skipped on re-runs
    3. silver.hr_employee     — configurable batch (default 80) every run

ENHANCEMENTS OVER v1
════════════════════
  1. Configurable batch size via EMPLOYEES_PER_RUN constant
  2. Reproducible runs via optional RANDOM_SEED
  3. Realistic email uniqueness — deduplication loop prevents collisions
  4. Realistic org-hierarchy — managers are assigned ONLY from same/higher dept
  5. Coaches are job-relevant (same department, different employee)
  6. Department manager back-fill after employee batch is written
  7. Weighted location distribution respects dept type (IT → more Remote)
  8. Permit/Visa logic is consistent — visa fields only set when permit exists
  9. hire_date is always ≤ updated_at ≤ now (no future dates)
 10. emergency_contact and emergency_phone are always in sync (both or neither)
 11. Duplicate name guard — adds numeric suffix if full_name already used this run
 12. Explicit schema alignment — every Row field matches silver DDL exactly
 13. study_school is department-relevant (IT → technical universities)
 14. Birthday constraint — age is realistically 22–60 (not 1970-2000 flat range)
 15. Proper logging with run summary printed at the end
 16. load_tracking update after each table write (mirrors the MERGE pipeline)
 17. Error-safe table existence check using information_schema
"""

import random
import logging
from datetime import datetime, timedelta, date

from pyspark.sql import SparkSession, Row
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType,
    BooleanType, TimestampType,
)

# ─────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────
CATALOG            = "smart_odoo"
SCHEMA             = "silver"
SOURCE_DB          = "python_sim"
COMPANY_ID         = 1
COMPANY            = "Smart Odoo LLC"
EMPLOYEES_PER_RUN  = 80        # change to any number
RANDOM_SEED        = None      # set to an int (e.g. 42) for reproducible runs

# ─────────────────────────────────────────────────────────────
# LOGGING
# ─────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-7s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("sim_hr")

# ─────────────────────────────────────────────────────────────
# SEED
# ─────────────────────────────────────────────────────────────
if RANDOM_SEED is not None:
    random.seed(RANDOM_SEED)
    log.info(f"Random seed fixed at {RANDOM_SEED}")

# ─────────────────────────────────────────────────────────────
# SPARK
# ─────────────────────────────────────────────────────────────
spark = SparkSession.builder.appName("sim_hr_delta_v2").getOrCreate()
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

now = datetime.now()

# ─────────────────────────────────────────────────────────────
# DATE HELPERS
# ─────────────────────────────────────────────────────────────
START_DATE = datetime(2023, 1, 1)
END_DATE   = datetime(2026, 4, 22)

def rand_date(start: datetime = START_DATE, end: datetime = END_DATE) -> datetime:
    return start + timedelta(days=random.randint(0, (end - start).days))

def rand_birthday(min_age: int = 22, max_age: int = 60) -> datetime:
    """
    Returns a birthday such that the employee is between min_age and max_age
    years old today — far more realistic than a flat 1970-2000 range.
    """
    today      = date.today()
    latest_dob = today.replace(year=today.year - min_age)   # youngest employee
    earliest   = today.replace(year=today.year - max_age)   # oldest employee
    delta_days = (latest_dob - earliest).days
    dob        = earliest + timedelta(days=random.randint(0, delta_days))
    return datetime(dob.year, dob.month, dob.day)

def maybe(val, prob_none: float = 0.3):
    """Return val or None based on prob_none probability."""
    return None if random.random() < prob_none else val

# ─────────────────────────────────────────────────────────────
# DELTA HELPERS
# ─────────────────────────────────────────────────────────────
def table_exists(table: str) -> bool:
    """Check via information_schema — works on Unity Catalog."""
    result = spark.sql(f"""
        SELECT COUNT(*) AS cnt
        FROM {CATALOG}.information_schema.tables
        WHERE table_schema = '{SCHEMA}' AND table_name = '{table}'
    """).collect()[0]["cnt"]
    return result > 0

def existing_values(table: str, col: str) -> set:
    """Return the set of existing values in a column; empty set if table missing."""
    if not table_exists(table):
        return set()
    return {r[col] for r in spark.sql(
        f"SELECT {col} FROM {CATALOG}.{SCHEMA}.{table}"
    ).collect()}

def max_id(table: str, col: str) -> int:
    """Return current MAX(col) or 0 if table is empty / missing."""
    if not table_exists(table):
        return 0
    result = spark.sql(
        f"SELECT COALESCE(MAX({col}), 0) AS m FROM {CATALOG}.{SCHEMA}.{table}"
    ).collect()[0]["m"]
    return result

def append_delta(df, table: str) -> int:
    """Append a DataFrame to a Delta table; return row count written."""
    count = df.count()
    (df.write
       .format("delta")
       .mode("append")
       .saveAsTable(f"{CATALOG}.{SCHEMA}.{table}"))
    log.info(f"  ✅  {table}  (+{count} rows)")
    return count

def update_load_tracking(table_name: str):
    """
    Update silver.load_tracking so the incremental MERGE pipeline
    won't re-process simulation rows on the next real ETL run.
    Safe no-op if load_tracking doesn't exist.
    """
    if not table_exists("load_tracking"):
        return
    spark.sql(f"""
        MERGE INTO {CATALOG}.{SCHEMA}.load_tracking AS t
        USING (SELECT '{table_name}' AS table_name, current_timestamp() AS last_write_date) AS s
        ON t.table_name = s.table_name
        WHEN MATCHED THEN UPDATE SET t.last_write_date = s.last_write_date
        WHEN NOT MATCHED THEN INSERT *
    """)

# ─────────────────────────────────────────────────────────────
# REFERENCE DATA
# ─────────────────────────────────────────────────────────────
DEPT_SEED = [
    (1, "IT",         "Technology & Infrastructure"),
    (2, "Sales",      "Revenue & Clients"),
    (3, "Finance",    "Accounting & Money"),
    (4, "HR",         "Human Resources"),
    (5, "Operations", "Logistics & Execution"),
]
DEPTS = {d[0]: d[1] for d in DEPT_SEED}

JOB_SEED = [
    (1,  "IT Manager",          1, "Lead & oversee IT infrastructure"),
    (2,  "Backend Developer",   1, "Build server-side applications"),
    (3,  "Data Analyst",        1, "Analyze data and generate insights"),
    (4,  "Sales Manager",       2, "Manage sales team and targets"),
    (5,  "Sales Executive",     2, "Drive client acquisition"),
    (6,  "Accountant",          3, "Manage financial records"),
    (7,  "Finance Analyst",     3, "Analyze financial performance"),
    (8,  "HR Manager",          4, "Oversee HR operations"),
    (9,  "Recruiter",           4, "Source and hire talent"),
    (10, "Operations Officer",  5, "Coordinate logistics and execution"),
]
JOBS      = {j[0]: (j[1], j[2]) for j in JOB_SEED}   # id → (name, dept_id)
DEPT_JOBS = {}
for jid, jname, did, _ in JOB_SEED:
    DEPT_JOBS.setdefault(did, []).append(jid)

# Manager-level job IDs — only these employees can be assigned as managers
MANAGER_JOB_IDS = {1, 4, 8, 10}   # IT Manager, Sales Manager, HR Manager, Operations Officer

CONTRACT_TYPES = {1: "Full-Time", 2: "Part-Time", 3: "Contract"}
CT_POOL        = [1] * 70 + [2] * 15 + [3] * 15

STUDY_FIELDS = {
    1: ["Computer Science", "Information Technology", "Software Engineering", "Data Science"],
    2: ["Business Administration", "Marketing", "Economics"],
    3: ["Accounting", "Finance", "Economics", "Business Administration"],
    4: ["Human Resources", "Psychology", "Business Administration"],
    5: ["Logistics", "Supply Chain", "Operations Management"],
}

# Department-relevant schools — IT/Data gets technical universities
SCHOOLS_BY_DEPT = {
    1: ["Cairo University (Faculty of Computers)", "Ain Shams University (Faculty of Engineering)",
        "German University in Cairo", "Nile University"],
    2: ["Cairo University (Faculty of Commerce)", "American University in Cairo",
        "Ain Shams University (Faculty of Commerce)", "Helwan University"],
    3: ["Cairo University (Faculty of Commerce)", "Ain Shams University (Faculty of Commerce)",
        "Alexandria University", "Mansoura University"],
    4: ["Cairo University", "Ain Shams University", "Helwan University", "Mansoura University"],
    5: ["Arab Academy for Science", "Helwan University", "Zagazig University", "Mansoura University"],
}

FIRST_NAMES = [
    "Ahmed", "Omar", "Mohamed", "Ali", "Youssef", "Kareem", "Hassan",
    "Ibrahim", "Mahmoud", "Khaled", "Tamer", "Walid", "Hossam", "Sherif",
    "Nour", "Sara", "Mariam", "Aya", "Dina", "Rania", "Hana", "Yasmin",
    "Mona", "Amira", "Fatma", "Noha", "Samar", "Tarek", "Bassem", "Wael",
]
LAST_NAMES = [
    "Hassan", "Ali", "Mohamed", "Said", "Younis", "Fathy", "Nabil",
    "Mahmoud", "Saleh", "Mostafa", "Abdallah", "Ramadan", "Khalil",
    "Mansour", "Ragab", "Gaber", "Zaki", "Fahmy", "Shawky", "Badawi",
    "Aziz", "Hamdy", "Sorour", "Kamal", "Fouad",
]

CITIES = [
    "Cairo", "Alexandria", "Giza", "Mansoura", "Tanta",
    "Aswan", "Luxor", "Suez", "Port Said", "Ismailia",
]

# Location weights differ by department
LOCATION_BY_DEPT = {
    1: ["Remote"] * 40 + ["Office"] * 45 + ["Field"] * 10 + [None] * 5,   # IT → more Remote
    2: ["Office"] * 35 + ["Field"]  * 40 + ["Remote"] * 20 + [None] * 5,  # Sales → more Field
    3: ["Office"] * 70 + ["Remote"] * 20 + ["Field"]  * 5  + [None] * 5,  # Finance → mostly Office
    4: ["Office"] * 75 + ["Remote"] * 15 + ["Field"]  * 5  + [None] * 5,  # HR → mostly Office
    5: ["Field"]  * 50 + ["Office"] * 30 + ["Remote"] * 15 + [None] * 5,  # Ops → mostly Field
}

LANGS  = ["ar_EG"] * 60 + ["en_US"] * 40
CERTS  = ["bachelor"] * 50 + ["master"] * 20 + ["doctor"] * 5 + ["other"] * 25

EMERGENCY_FIRST = ["Maged", "Rasha", "Sameh", "Dalia", "Hany", "Ghada", "Essam", "Iman"]


def make_email(name: str, used_emails: set) -> str:
    """
    Generate work_email from name; if collision, append numeric suffix.
    Ensures every employee in the run has a unique email.
    """
    base = name.lower().replace(" ", ".") + "@company.com"
    if base not in used_emails:
        return base
    idx = 1
    while True:
        candidate = name.lower().replace(" ", ".") + f"{idx}@company.com"
        if candidate not in used_emails:
            return candidate
        idx += 1

def make_phone() -> str:
    prefix = random.choice(["10", "11", "12"])
    return f"+20 1{prefix}{random.randint(10000000, 99999999)}"


# ═══════════════════════════════════════════════════════════════
# 1. HR_DEPARTMENT  (5 fixed — seed once)
# ═══════════════════════════════════════════════════════════════
dept_schema = StructType([
    StructField("hr_department_id",       IntegerType(),   True),
    StructField("company_id",             IntegerType(),   True),
    StructField("company_name",           StringType(),    True),
    StructField("parent_id",              IntegerType(),   True),
    StructField("parent_name",            StringType(),    True),
    StructField("manager_id",             IntegerType(),   True),
    StructField("manager_name",           StringType(),    True),
    StructField("master_department_id",   IntegerType(),   True),
    StructField("master_department_name", StringType(),    True),
    StructField("name",                   StringType(),    True),
    StructField("parent_path",            StringType(),    True),
    StructField("note",                   StringType(),    True),
    StructField("active",                 BooleanType(),   True),
    StructField("created_at",             TimestampType(), True),
    StructField("updated_at",             TimestampType(), True),
    StructField("dwh_loaded_at",          TimestampType(), True),
    StructField("dwh_source_db",          StringType(),    True),
])

existing_dept_names = existing_values("hr_department", "name")
dept_rows = []
for did, dname, dnote in DEPT_SEED:
    if dname in existing_dept_names:
        continue
    dept_rows.append(Row(
        hr_department_id       = did,
        company_id             = COMPANY_ID,
        company_name           = COMPANY,
        parent_id              = None,
        parent_name            = None,
        manager_id             = None,   # back-filled after employees are written
        manager_name           = None,
        master_department_id   = did,
        master_department_name = dname,
        name                   = dname,
        parent_path            = f"/{did}/",
        note                   = dnote,
        active                 = True,
        created_at             = now,
        updated_at             = now,
        dwh_loaded_at          = now,
        dwh_source_db          = SOURCE_DB,
    ))

if dept_rows:
    append_delta(spark.createDataFrame(dept_rows, schema=dept_schema), "hr_department")
    update_load_tracking("hr_department")
else:
    log.info("  ⏭   hr_department  (already seeded)")


# ═══════════════════════════════════════════════════════════════
# 2. HR_JOB  (10 fixed — seed once)
# ═══════════════════════════════════════════════════════════════
job_schema = StructType([
    StructField("hr_job_id",           IntegerType(),   True),
    StructField("company_id",          IntegerType(),   True),
    StructField("company_name",        StringType(),    True),
    StructField("department_id",       IntegerType(),   True),
    StructField("department_name",     StringType(),    True),
    StructField("user_id",             IntegerType(),   True),
    StructField("user_name",           StringType(),    True),
    StructField("contract_type_id",    IntegerType(),   True),
    StructField("contract_type_name",  StringType(),    True),
    StructField("sequence",            IntegerType(),   True),
    StructField("no_of_recruitment",   IntegerType(),   True),
    StructField("name",                StringType(),    True),
    StructField("description",         StringType(),    True),
    StructField("requirements",        StringType(),    True),
    StructField("active",              BooleanType(),   True),
    StructField("created_at",          TimestampType(), True),
    StructField("updated_at",          TimestampType(), True),
    StructField("dwh_loaded_at",       TimestampType(), True),
    StructField("dwh_source_db",       StringType(),    True),
])

existing_job_names = existing_values("hr_job", "name")
job_rows = []
for jid, jname, jdept, jdesc in JOB_SEED:
    if jname in existing_job_names:
        continue
    ct_id = random.choice(CT_POOL)
    uid   = random.randint(1, 10)
    job_rows.append(Row(
        hr_job_id          = jid,
        company_id         = COMPANY_ID,
        company_name       = COMPANY,
        department_id      = jdept,
        department_name    = DEPTS[jdept],
        user_id            = uid,
        user_name          = f"User_{uid}",
        contract_type_id   = ct_id,
        contract_type_name = CONTRACT_TYPES[ct_id],
        sequence           = jid,
        no_of_recruitment  = random.randint(1, 5),
        name               = jname,
        description        = jdesc,
        requirements       = maybe(f"Min 2 years experience in {jname}", 0.4),
        active             = True,
        created_at         = now,
        updated_at         = now,
        dwh_loaded_at      = now,
        dwh_source_db      = SOURCE_DB,
    ))

if job_rows:
    append_delta(spark.createDataFrame(job_rows, schema=job_schema), "hr_job")
    update_load_tracking("hr_job")
else:
    log.info("  ⏭   hr_job  (already seeded)")


# ═══════════════════════════════════════════════════════════════
# 3. HR_EMPLOYEE  — EMPLOYEES_PER_RUN new rows every run
# ═══════════════════════════════════════════════════════════════
emp_id_start = max_id("hr_employee", "hr_employee_id") + 1
log.info(f"Starting hr_employee_id from : {emp_id_start}")

# Load existing employees for parent/coach references
# Structure: { emp_id: {"name": str, "dept_id": int, "job_id": int} }
try:
    existing_emps = {
        r.hr_employee_id: {
            "name":    r.name,
            "dept_id": r.department_id if hasattr(r, "department_id") else None,
        }
        for r in spark.sql(f"""
            SELECT e.hr_employee_id, e.name,
                   j.department_id
            FROM   {CATALOG}.{SCHEMA}.hr_employee e
            LEFT   JOIN {CATALOG}.{SCHEMA}.hr_job j
                   ON  e.job_id = j.hr_job_id
        """).collect()
    }
except Exception:
    existing_emps: dict = {}

emp_schema = StructType([
    StructField("hr_employee_id",                IntegerType(),   True),
    StructField("company_id",                    IntegerType(),   True),
    StructField("company_name",                  StringType(),    True),
    StructField("user_id",                       IntegerType(),   True),
    StructField("user_name",                     StringType(),    True),
    StructField("parent_id",                     IntegerType(),   True),
    StructField("parent_name",                   StringType(),    True),
    StructField("coach_id",                      IntegerType(),   True),
    StructField("coach_name",                    StringType(),    True),
    StructField("work_contact_id",               IntegerType(),   True),
    StructField("work_contact_name",             StringType(),    True),
    StructField("name",                          StringType(),    True),
    StructField("legal_name",                    StringType(),    True),
    StructField("work_phone",                    StringType(),    True),
    StructField("mobile_phone",                  StringType(),    True),
    StructField("work_email",                    StringType(),    True),
    StructField("lang",                          StringType(),    True),
    StructField("certificate",                   StringType(),    True),
    StructField("study_field",                   StringType(),    True),
    StructField("study_school",                  StringType(),    True),
    StructField("place_of_birth",                StringType(),    True),
    StructField("emergency_contact",             StringType(),    True),
    StructField("emergency_phone",               StringType(),    True),
    StructField("permit_no",                     StringType(),    True),
    StructField("visa_no",                       StringType(),    True),
    StructField("barcode",                       StringType(),    True),
    StructField("today_location_name",           StringType(),    True),
    StructField("salary_distribution",           StringType(),    True),
    StructField("employee_properties",           StringType(),    True),
    StructField("active",                        BooleanType(),   True),
    StructField("birthday_public_display",       BooleanType(),   True),
    StructField("work_permit_scheduled_activity", BooleanType(),  True),
    StructField("birthday",                      TimestampType(), True),
    StructField("visa_expire",                   TimestampType(), True),
    StructField("work_permit_expiration_date",   TimestampType(), True),
    StructField("created_at",                    TimestampType(), True),
    StructField("updated_at",                    TimestampType(), True),
    StructField("dwh_loaded_at",                 TimestampType(), True),
    StructField("dwh_source_db",                 StringType(),    True),
])

emp_rows:          list[Row]   = []
used_full_names:   set[str]    = set()
used_emails:       set[str]    = set()
emp_id:            int         = emp_id_start

# Track employees generated this run: id → {name, dept_id, is_manager}
this_run: dict[int, dict] = {}

for _ in range(EMPLOYEES_PER_RUN):

    # ── Department & Job ──────────────────────────────────────
    dept_id   = random.choice(list(DEPTS.keys()))
    dept_name = DEPTS[dept_id]
    job_id    = random.choice(DEPT_JOBS[dept_id])
    job_name  = JOBS[job_id][0]
    is_mgr    = job_id in MANAGER_JOB_IDS

    # ── Name (with collision guard) ───────────────────────────
    attempts = 0
    while True:
        full_name = f"{random.choice(FIRST_NAMES)} {random.choice(LAST_NAMES)}"
        if full_name not in used_full_names:
            break
        attempts += 1
        if attempts > 20:
            # fallback: append suffix to guarantee uniqueness
            full_name = f"{full_name} {emp_id}"
            break
    used_full_names.add(full_name)

    # ── Email (unique) ────────────────────────────────────────
    email = make_email(full_name, used_emails)
    used_emails.add(email)

    # ── Phone ─────────────────────────────────────────────────
    phone  = make_phone()
    mobile = phone if random.random() < 0.7 else make_phone()

    # ── Manager (parent_id) ───────────────────────────────────
    # Only assign managers from same department who hold a manager-level job.
    # Falls back to cross-dept manager if none available in dept.
    candidate_mgrs = [
        eid for eid, info in {**existing_emps, **this_run}.items()
        if info.get("is_mgr") and info.get("dept_id") == dept_id and eid != emp_id
    ]
    if not candidate_mgrs:
        # fallback to any manager
        candidate_mgrs = [
            eid for eid, info in {**existing_emps, **this_run}.items()
            if info.get("is_mgr") and eid != emp_id
        ]

    if candidate_mgrs and random.random() < 0.75:
        parent_id   = random.choice(candidate_mgrs)
        parent_name = {**existing_emps, **this_run}[parent_id]["name"]
    else:
        parent_id, parent_name = None, None

    # ── Coach ─────────────────────────────────────────────────
    # Coach is any employee in the same department (not self, not manager)
    candidate_coaches = [
        eid for eid, info in {**existing_emps, **this_run}.items()
        if info.get("dept_id") == dept_id and eid != emp_id and eid != parent_id
    ]
    if not candidate_coaches:
        candidate_coaches = [
            eid for eid, info in {**existing_emps, **this_run}.items()
            if eid != emp_id
        ]

    if candidate_coaches and random.random() < 0.6:
        coach_id   = random.choice(candidate_coaches)
        coach_name = {**existing_emps, **this_run}[coach_id]["name"]
    else:
        coach_id, coach_name = None, None

    # ── Work contact ──────────────────────────────────────────
    wc_id   = emp_id if random.random() < 0.8 else None
    wc_name = full_name if wc_id else None

    # ── Education ─────────────────────────────────────────────
    cert        = random.choice(CERTS)
    study_field = random.choice(STUDY_FIELDS.get(dept_id, ["General Studies"]))
    school      = random.choice(SCHOOLS_BY_DEPT.get(dept_id, ["Cairo University"]))

    # ── Work permit / visa (consistent block) ─────────────────
    has_permit = random.random() < 0.25
    if has_permit:
        permit_no  = f"PRM-{random.randint(100000, 999999)}"
        visa_no    = f"VIS-{random.randint(100000, 999999)}"
        visa_exp   = maybe(now + timedelta(days=random.randint(30, 730)), 0.3)
        wp_exp     = maybe(now + timedelta(days=random.randint(30, 730)), 0.3)
        wp_sched   = random.random() < 0.15
    else:
        permit_no = visa_no = visa_exp = wp_exp = None
        wp_sched  = False

    # ── Emergency contact (both or neither) ───────────────────
    has_ec    = random.random() > 0.3
    ec_name   = (f"{random.choice(EMERGENCY_FIRST)} {random.choice(LAST_NAMES)}"
                 if has_ec else None)
    ec_phone  = make_phone() if has_ec else None

    # ── Hire date & timestamps ─────────────────────────────────
    hire_date   = rand_date(START_DATE, now)    # always in the past
    # updated_at is >= hire_date and <= now
    update_days = (now - hire_date).days
    updated_at  = (hire_date + timedelta(days=random.randint(0, max(update_days, 0))))

    # ── Location (department-weighted) ────────────────────────
    location = random.choice(LOCATION_BY_DEPT.get(dept_id, ["Office"] * 60 + [None] * 40))

    # ── Build Row ─────────────────────────────────────────────
    emp_rows.append(Row(
        hr_employee_id                   = emp_id,
        company_id                       = COMPANY_ID,
        company_name                     = COMPANY,
        user_id                          = emp_id,
        user_name                        = full_name,
        parent_id                        = parent_id,
        parent_name                      = parent_name,
        coach_id                         = coach_id,
        coach_name                       = coach_name,
        work_contact_id                  = wc_id,
        work_contact_name                = wc_name,
        name                             = full_name,
        legal_name                       = full_name if random.random() < 0.8 else None,
        work_phone                       = phone,
        mobile_phone                     = mobile,
        work_email                       = email,
        lang                             = random.choice(LANGS),
        certificate                      = cert,
        study_field                      = study_field,
        study_school                     = school,
        place_of_birth                   = random.choice(CITIES),
        emergency_contact                = ec_name,
        emergency_phone                  = ec_phone,
        permit_no                        = permit_no,
        visa_no                          = visa_no,
        barcode                          = f"EMP-{emp_id:06d}",
        today_location_name              = location,
        salary_distribution              = None,
        employee_properties              = None,
        active                           = random.random() < 0.9,
        birthday_public_display          = random.random() < 0.6,
        work_permit_scheduled_activity   = wp_sched,
        birthday                         = rand_birthday(),
        visa_expire                      = visa_exp,
        work_permit_expiration_date      = wp_exp,
        created_at                       = hire_date,
        updated_at                       = updated_at,
        dwh_loaded_at                    = now,
        dwh_source_db                    = SOURCE_DB,
    ))

    # Register in this_run lookup
    this_run[emp_id] = {"name": full_name, "dept_id": dept_id, "is_mgr": is_mgr}
    emp_id += 1

df_emp = spark.createDataFrame(emp_rows, schema=emp_schema)
append_delta(df_emp, "hr_employee")
update_load_tracking("hr_employee")


# ═══════════════════════════════════════════════════════════════
# 4. BACK-FILL department managers
#    After writing employees, assign one manager per department
#    using only employees holding a manager-level job.
# ═══════════════════════════════════════════════════════════════
log.info("Back-filling department managers...")

all_emps = {**existing_emps, **this_run}
dept_backfill_count = 0

for dept_id, dept_name in DEPTS.items():
    # Check if department already has a manager assigned
    try:
        current_mgr = spark.sql(f"""
            SELECT manager_id
            FROM {CATALOG}.{SCHEMA}.hr_department
            WHERE hr_department_id = {dept_id}
        """).collect()
        if current_mgr and current_mgr[0]["manager_id"] is not None:
            continue  # already has a manager, skip
    except Exception:
        continue

    # Find a manager-level employee from this department
    dept_mgrs = [
        eid for eid, info in this_run.items()
        if info.get("dept_id") == dept_id and info.get("is_mgr")
    ]
    if not dept_mgrs:
        continue

    chosen_id   = random.choice(dept_mgrs)
    chosen_name = this_run[chosen_id]["name"]

    spark.sql(f"""
        UPDATE {CATALOG}.{SCHEMA}.hr_department
        SET manager_id   = {chosen_id},
            manager_name = '{chosen_name}',
            updated_at   = current_timestamp(),
            dwh_loaded_at = current_timestamp()
        WHERE hr_department_id = {dept_id}
          AND manager_id IS NULL
    """)
    log.info(f"  ✅  hr_department.{dept_name} → manager = {chosen_name} (id={chosen_id})")
    dept_backfill_count += 1


# ═══════════════════════════════════════════════════════════════
# SUMMARY
# ═══════════════════════════════════════════════════════════════
managers_in_batch  = sum(1 for info in this_run.values() if info.get("is_mgr"))
active_in_batch    = sum(1 for r in emp_rows if r.active)
with_permit        = sum(1 for r in emp_rows if r.permit_no is not None)
remote_count       = sum(1 for r in emp_rows if r.today_location_name == "Remote")

log.info("═" * 50)
log.info(f"  Run complete — {EMPLOYEES_PER_RUN} employees written")
log.info(f"  IDs          : {emp_id_start} → {emp_id - 1}")
log.info(f"  Managers     : {managers_in_batch}")
log.info(f"  Active       : {active_in_batch} / {EMPLOYEES_PER_RUN}")
log.info(f"  With permit  : {with_permit}")
log.info(f"  Remote       : {remote_count}")
log.info(f"  Dept managers back-filled : {dept_backfill_count}")
log.info("═" * 50)